In [1]:
import numpy as np
import pandas as pd

In [2]:
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

In [3]:
def overview(df):
    pd.set_option('display.max_rows', None)
    print("(Broj_redova, broj_kolona): ", df.shape)
    print("Tip podataka svake kolone:\n", df.dtypes.to_frame(name='Tip_podatka'))
    missing = pd.DataFrame({'Broj_NA': df.isna().sum(),'Procenat_NA': round(df.isna().sum()/len(df)*100,2)}).sort_values(by='Procenat_NA',ascending=False)
    print("NA vrednosti:\n", missing)
overview(train)
overview(test)

(Broj_redova, broj_kolona):  (59381, 128)
Tip podataka svake kolone:
                     Tip_podatka
Id                        int64
Product_Info_1            int64
Product_Info_2           object
Product_Info_3            int64
Product_Info_4          float64
Product_Info_5            int64
Product_Info_6            int64
Product_Info_7            int64
Ins_Age                 float64
Ht                      float64
Wt                      float64
BMI                     float64
Employment_Info_1       float64
Employment_Info_2         int64
Employment_Info_3         int64
Employment_Info_4       float64
Employment_Info_5         int64
Employment_Info_6       float64
InsuredInfo_1             int64
InsuredInfo_2             int64
InsuredInfo_3             int64
InsuredInfo_4             int64
InsuredInfo_5             int64
InsuredInfo_6             int64
InsuredInfo_7             int64
Insurance_History_1       int64
Insurance_History_2       int64
Insurance_History_3       int64
In

In [4]:
test_id = test['Id']

test_id.to_csv("..\\rezultati\\test_id.csv", index=False)

In [5]:
train = train.drop('Id', axis=1)
test = test.drop('Id', axis=1)

In [6]:
target = "Response"

cat_cols = [
    "Product_Info_1","Product_Info_2","Product_Info_3","Product_Info_5","Product_Info_6","Product_Info_7",
    "Employment_Info_2","Employment_Info_3","Employment_Info_5",
    "InsuredInfo_1","InsuredInfo_2","InsuredInfo_3","InsuredInfo_4","InsuredInfo_5","InsuredInfo_6","InsuredInfo_7",
    "Insurance_History_1","Insurance_History_2","Insurance_History_3","Insurance_History_4","Insurance_History_7",
    "Insurance_History_8","Insurance_History_9","Family_Hist_1",
    "Medical_History_2","Medical_History_3","Medical_History_4","Medical_History_5","Medical_History_6","Medical_History_7",
    "Medical_History_8","Medical_History_9","Medical_History_11","Medical_History_12","Medical_History_13","Medical_History_14",
    "Medical_History_16","Medical_History_17","Medical_History_18","Medical_History_19","Medical_History_20","Medical_History_21",
    "Medical_History_22","Medical_History_23","Medical_History_25","Medical_History_26","Medical_History_27","Medical_History_28",
    "Medical_History_29","Medical_History_30","Medical_History_31","Medical_History_33","Medical_History_34","Medical_History_35",
    "Medical_History_36","Medical_History_37","Medical_History_38","Medical_History_39","Medical_History_40","Medical_History_41"
]

continuous_cols = [
    "Product_Info_4","Ins_Age","Ht","Wt","BMI",
    "Employment_Info_1","Employment_Info_4","Employment_Info_6",
    "Insurance_History_5",
    "Family_Hist_2","Family_Hist_3","Family_Hist_4","Family_Hist_5"
]

discrete_cols = [
    "Medical_History_1","Medical_History_10","Medical_History_15",
    "Medical_History_24","Medical_History_32"
]

dummy_cols = [f"Medical_Keyword_{i}" for i in range(1,49)]

Kontinualne varijable ucitane su kao float64 ili int64, sto je Python dobro prepoznao, samim tim suvisno je pretvarati ih u ovom trenutku.  

Gledajuci EDA, moze se uociti da sve varijable koje predstavljaju diskretne tipove podataka, imaju veliki broj razlicitih vrednosti(priblizno 90-250), u odnosu na ostale atribute gde se broj jedinstvenih vrednosti krece izmedju 2-3. Sa obzirom na to, da se odnose na medicinsku istoriju pacijenta, ove promenljive nema smisla tretirati kao numericke koje uzimaju vrednosti iz prebrojivog skupa.  

Dummy promenljive učitane su kao int64 i sadrže binarne vrednosti (0 i 1), čime već predstavljaju odgovarajući numerički oblik pogodan za primenu algoritama mašinskog učenja. S obzirom na to da su ove promenljive unapred kodirane u obliku indikatora prisustva ili odsustva određene osobine, nije postojala potreba za dodatnom promenom njihovog tipa podataka.  

Varijable nominalnog tipa potrebno je prevesti u odgovarajući tip podataka, kako ih sistem u nastavku analize ne bi tretirao kao numeričke promenljive. Ovakve promenljive nemaju prirodan redosled niti matematički smislen odnos između kategorija, pa njihovo pogrešno tumačenje može dovesti do formiranja nepostojećih odnosa među vrednostima.

Kako bi atribute numerickog tipa bilo moguce pretvoriti u atribute kategorijskog tipa, potrebno je videti pre svega, da li ne postoje vrednosti poput
"None","Missing" itd... i ukoliko postoje proglasiti ih pre svega za NA, kako od ovakvih vrednosti sistem ne bi kreirao nove kategorije. Sa obzirom, da su od strane sistema, sve promenljive osim Product_Info_2 prepoznate kao int, samo ce se za navedenu promenljivu proveriti ovaj slucaj.

In [7]:
train['Product_Info_2'].value_counts(dropna=True)

Product_Info_2
D3    14321
D4    10812
A8     6835
D1     6554
D2     6286
E1     2647
A1     2363
A6     2098
A2     1974
A7     1383
B2     1122
A3      977
A5      775
C3      306
C1      285
C4      219
A4      210
C2      160
B1       54
Name: count, dtype: int64

In [8]:
test['Product_Info_2'].value_counts(dropna=True)
    

Product_Info_2
D3    4432
D4    3259
A8    2305
D2    2058
D1    2057
A2    1098
E1    1064
A1     856
A6     635
A3     587
A7     440
B2     324
A5     234
C3     131
C1      92
C4      72
A4      53
C2      37
B1      31
Name: count, dtype: int64

In [9]:
def numeric_to_categorical(df, columns):
    for col in columns:
        df[col] = df[col].astype('category')
    
    return df

In [10]:
train = numeric_to_categorical(train, cat_cols)
test = numeric_to_categorical(test, cat_cols)

In [11]:
overview(train)
overview(test)

(Broj_redova, broj_kolona):  (59381, 127)
Tip podataka svake kolone:
                     Tip_podatka
Product_Info_1         category
Product_Info_2         category
Product_Info_3         category
Product_Info_4          float64
Product_Info_5         category
Product_Info_6         category
Product_Info_7         category
Ins_Age                 float64
Ht                      float64
Wt                      float64
BMI                     float64
Employment_Info_1       float64
Employment_Info_2      category
Employment_Info_3      category
Employment_Info_4       float64
Employment_Info_5      category
Employment_Info_6       float64
InsuredInfo_1          category
InsuredInfo_2          category
InsuredInfo_3          category
InsuredInfo_4          category
InsuredInfo_5          category
InsuredInfo_6          category
InsuredInfo_7          category
Insurance_History_1    category
Insurance_History_2    category
Insurance_History_3    category
Insurance_History_4    category
In

Uocava se da train i test skup za iste atribute imaju priblizan procenat nedostajucih vrednosti. Bez dodatnih razmatranja, kolone Medical_History_10, Medical_History_32 i Medical_History_24 izdvojene su kao kandidati za izbacivanje, s obzirom na to da procenat nedostajucih vrednosti prelazi 90%. Ovako visok nivo nedostajucih podataka znacajno ogranicava kolicinu raspolozivih informacija, dok bi imputacija velikog broja vrednosti mogla dovesti do unosenja suma i smanjenja pouzdanosti modela.  Sa druge strane, Medical_History_15 ima 75% nedostajucih  vrednosti, ali pored toga, ima ~250 razlicitih kategorija zbog cega smatram da je takodje validan kandidat za izbacivanje. Ostale nedostajuce vrednosti, bice razmotrene kasnije.

In [12]:
train= train.drop([ 'Medical_History_10',
    'Medical_History_32',
    'Medical_History_24', 
       'Medical_History_15'], axis=1)
test= test.drop( ['Medical_History_10',
    'Medical_History_32',
    'Medical_History_24',
     'Medical_History_15'], axis=1)

In [13]:
print(train.shape)
print(test.shape)

(59381, 123)
(19765, 122)


In [14]:
for col in cat_cols:
    print(col, "->", train[col].nunique())
   

Product_Info_1 -> 2
Product_Info_2 -> 19
Product_Info_3 -> 34
Product_Info_5 -> 2
Product_Info_6 -> 2
Product_Info_7 -> 3
Employment_Info_2 -> 36
Employment_Info_3 -> 2
Employment_Info_5 -> 2
InsuredInfo_1 -> 3
InsuredInfo_2 -> 2
InsuredInfo_3 -> 11
InsuredInfo_4 -> 2
InsuredInfo_5 -> 2
InsuredInfo_6 -> 2
InsuredInfo_7 -> 2
Insurance_History_1 -> 2
Insurance_History_2 -> 3
Insurance_History_3 -> 3
Insurance_History_4 -> 3
Insurance_History_7 -> 3
Insurance_History_8 -> 3
Insurance_History_9 -> 3
Family_Hist_1 -> 3
Medical_History_2 -> 579
Medical_History_3 -> 3
Medical_History_4 -> 2
Medical_History_5 -> 3
Medical_History_6 -> 3
Medical_History_7 -> 3
Medical_History_8 -> 3
Medical_History_9 -> 3
Medical_History_11 -> 3
Medical_History_12 -> 3
Medical_History_13 -> 3
Medical_History_14 -> 3
Medical_History_16 -> 3
Medical_History_17 -> 3
Medical_History_18 -> 3
Medical_History_19 -> 3
Medical_History_20 -> 3
Medical_History_21 -> 3
Medical_History_22 -> 2
Medical_History_23 -> 3
Medica

Promenljiva Medical_History_2 ima veliki broj razlicitih kategorija(579). Sa obzirom da je sa ovakvim brojem kategorija. tesko raditi, pogotovo u slucaju potrebe za vrsenjem one hot encoding-a, bice izvrsena transformacija.

In [15]:
train['Medical_History_2'].value_counts()

Medical_History_2
112    11027
162     8343
491     5797
335     3032
161     2942
16      2920
261     2303
478     1957
3       1442
628     1186
565     1001
155      974
145      907
613      752
366      701
387      644
132      481
156      464
434      439
453      413
181      375
528      329
276      318
217      275
202      267
57       260
600      259
14       258
52       247
48       222
278      214
125      206
228      206
610      192
33       190
407      184
260      174
140      165
192      156
418      142
350      142
412      140
564      137
373      133
160      133
575      126
561      124
515      123
354      123
403      117
264      117
43        97
627       92
82        88
280       84
42        80
413       78
379       76
255       73
579       71
459       69
230       69
7         69
631       69
245       68
298       68
441       66
265       65
35        64
570       64
324       62
122       59
446       55
108       52
415       52
633    

Moze se uociti da postoji veliki broj promenljivih koje dominiraju u broju pojavljivanja, pa je ova promenljiva pogodna za proglasenje svih promenljivih koje nemaju puno pujavljivanja, u 'Others', cime ce se smanjiti broj kategorija.

In [ ]:
train_final = train.copy()
test_final = test.copy()
print(train_final.select_dtypes(include='category').columns.tolist())
print(train_final.dtypes)
train_final.to_parquet("../preprocessed/train_final.parquet", index=False)
test_final.to_parquet("../preprocessed/test_final.parquet", index=False)

['Product_Info_1', 'Product_Info_2', 'Product_Info_3', 'Product_Info_5', 'Product_Info_6', 'Product_Info_7', 'Employment_Info_2', 'Employment_Info_3', 'Employment_Info_5', 'InsuredInfo_1', 'InsuredInfo_2', 'InsuredInfo_3', 'InsuredInfo_4', 'InsuredInfo_5', 'InsuredInfo_6', 'InsuredInfo_7', 'Insurance_History_1', 'Insurance_History_2', 'Insurance_History_3', 'Insurance_History_4', 'Insurance_History_7', 'Insurance_History_8', 'Insurance_History_9', 'Family_Hist_1', 'Medical_History_2', 'Medical_History_3', 'Medical_History_4', 'Medical_History_5', 'Medical_History_6', 'Medical_History_7', 'Medical_History_8', 'Medical_History_9', 'Medical_History_11', 'Medical_History_12', 'Medical_History_13', 'Medical_History_14', 'Medical_History_16', 'Medical_History_17', 'Medical_History_18', 'Medical_History_19', 'Medical_History_20', 'Medical_History_21', 'Medical_History_22', 'Medical_History_23', 'Medical_History_25', 'Medical_History_26', 'Medical_History_27', 'Medical_History_28', 'Medical_Hi